We process the original AIT-ADS in two stages to get the alert datasets in form appropriate for our Alert Grouping Pipeline implementation for this methodology.

# Stage 1

## Alert CSV Files

This stage gives us all alerts across the following **8 CSV files**:

* `fox_alerts.csv`
* `harrison_alerts.csv`
* `russellmitchell_alerts.csv`
* `santos_alerts.csv`
* `shaw_alerts.csv`
* `wardbeck_alerts.csv`
* `wheeler_alerts.csv`
* `wilson_alerts.csv`

# Stage 2

## Column Standardization

After loading the files, we rename the columns to match the requirements of the **alert grouping pipeline**:

| Original column | New column           |
| --------------- | -------------------- |
| `source`        | `name`               |
| `short`         | `short_abbreviation` |
| `name`          | `short`              |

## IP Column

We also create a new standardized `ip` column.

The value is selected using the following logic:

```text
if src_ip is not null:
    ip = src_ip
else:
    ip = agent_ip
```

In other words, **`src_ip` takes precedence**, and `agent_ip` is used as a fallback when `src_ip` is null.

This results in a consistent `ip` field that can be used downstream by the alert grouping pipeline.


### Below is the code block for stage 1

In [1]:
import json
import os
from datetime import datetime

import pytz
import yaml

import timestampExtractor
from abbrvs import get_short


# =========================================================
# AIT-ADS scenarios
# =========================================================

scenarios = [
    "russellmitchell",
    "fox",
    "harrison",
    "santos",
    "shaw",
    "wardbeck",
    "wheeler",
    "wilson",
]


# =========================================================
# Paths
# =========================================================

# JSON files are in the same directory as the notebook
alerts_json_path = "."

# Server configuration files
server_config_path = "./server_configs"

# Output directory
alerts_csv_path = "./alerts_csv"

os.makedirs(alerts_csv_path, exist_ok=True)


# =========================================================
# CSV columns
# =========================================================
#
# Required grouping fields:
#   time
#   name
#   host
#   short
#
# IP information:
#   agent_ip
#   src_ip
#
# Additional AIT-ADS/Wazuh/Suricata information:
#   source
#   rule_id
#   level
#   decoder
#   location
#   src_port
#   dst_ip
#   dst_port
#   protocol
#   alert_id
#
# No time_label or event_label.
# =========================================================

CSV_HEADER = (
    "time,"
    "source,"
    "name,"
    "short,"
    "agent_ip,"
    "host,"
    "rule_id,"
    "level,"
    "decoder,"
    "location,"
    "src_ip,"
    "src_port,"
    "dst_ip,"
    "dst_port,"
    "protocol,"
    "alert_id\n"
)


# =========================================================
# Helper
# =========================================================

def csv_value(value):
    """
    Convert a value to a CSV-safe string.

    Commas and newlines are replaced so that each alert
    remains exactly one CSV row.
    """

    if value is None:
        return ""

    return (
        str(value)
        .replace(",", " ")
        .replace("\n", " ")
        .replace("\r", " ")
    )


def write_alert(
    alerts_csv,
    log_time,
    source,
    name,
    short,
    agent_ip="",
    host="",
    rule_id="",
    level="",
    decoder="",
    location="",
    src_ip="",
    src_port="",
    dst_ip="",
    dst_port="",
    protocol="",
    alert_id="",
):
    """
    Write one normalized alert to the CSV file.
    """

    values = [
        int(log_time),
        source,
        name,
        short,
        agent_ip,
        host,
        rule_id,
        level,
        decoder,
        location,
        src_ip,
        src_port,
        dst_ip,
        dst_port,
        protocol,
        alert_id,
    ]

    alerts_csv.write(
        ",".join(csv_value(value) for value in values)
        + "\n"
    )


# =========================================================
# Process scenarios
# =========================================================

for scenario in scenarios:

    wazuhfile = scenario + "_wazuh.json"
    aminerfile = scenario + "_aminer.json"
    configfile = scenario + ".yaml"

    alertscsv = (
        alerts_csv_path
        + "/"
        + scenario
        + "_alerts.csv"
    )

    wazuh_path = alerts_json_path + "/" + wazuhfile
    aminer_path = alerts_json_path + "/" + aminerfile
    config_path = server_config_path + "/" + configfile

    print("Processing:", scenario)

    # =====================================================
    # Load server configuration
    # =====================================================

    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as config_in:

        server_config = yaml.safe_load(config_in)

    # -----------------------------------------------------
    # Build:
    #
    #     IP address -> hostname
    #
    # This is exactly the same mapping used by the
    # original AlertBERT preprocessing code.
    # -----------------------------------------------------

    ips = {}

    for hostname in server_config:

        ip = server_config[hostname]["default_ipv4_address"]

        ips[ip] = hostname

    # =====================================================
    # Open input/output files
    # =====================================================

    with open(
        wazuh_path,
        "r",
        encoding="utf-8",
    ) as wazuh_in, open(
        aminer_path,
        "r",
        encoding="utf-8",
    ) as aminer_in, open(
        alertscsv,
        "w",
        encoding="utf-8",
    ) as alerts_csv:

        # -------------------------------------------------
        # Write CSV header
        # -------------------------------------------------

        alerts_csv.write(CSV_HEADER)

        # =================================================
        # Wazuh / Suricata alerts
        # =================================================

        for line in wazuh_in:

            if "The average number" in line:
                continue

            j = json.loads(line)

            # -------------------------------------------------
            # Timestamp extraction
            # -------------------------------------------------

            if "timestamp" in j.get("data", {}):

                log_time = (
                    datetime.strptime(
                        j["data"]["timestamp"],
                        "%Y-%m-%dT%H:%M:%S.%f%z",
                    )
                    .replace(tzinfo=pytz.utc)
                    .timestamp()
                )

            elif "timestamp" in j.get("predecoder", {}):

                timestamp = j["predecoder"]["timestamp"]

                if "/" in timestamp:

                    log_time = (
                        datetime.strptime(
                            timestamp,
                            "%m/%d/%Y-%H:%M:%S.%f",
                        )
                        .replace(tzinfo=pytz.utc)
                        .timestamp()
                    )

                else:

                    log_time = (
                        datetime.strptime(
                            "2022 " + timestamp,
                            "%Y %b %d %H:%M:%S",
                        )
                        .replace(tzinfo=pytz.utc)
                        .timestamp()
                    )

            elif "full_log" in j:

                logline = j["full_log"]

                filename = j["location"].split("/")[-1]

                if filename.endswith("-access.log"):
                    filename = "-access.log"

                elif filename.endswith("-error.log"):
                    filename = "-error.log"

                elif filename.endswith("syslog"):
                    filename = "logs/syslog"

                else:
                    filename = "/" + filename

                try:

                    log_time = timestampExtractor.timestampExtractor[
                        filename
                    ](logline).timestamp()

                except KeyError:

                    print(
                        "Unknown timestamp format:",
                        filename,
                    )

                    continue

            else:

                print("Unknown alert format:", j)

                continue

            # -------------------------------------------------
            # Alert description
            # -------------------------------------------------

            description = (
                j.get("rule", {})
                .get("description", "")
            )

            # Keep original description for source detection
            if description.startswith("Suricata: "):
                source = "Suricata"
            else:
                source = "Wazuh"

            # Preserve AlertBERT abbreviation behavior
            short = get_short(description)

            # Preserve AlertBERT's Wazuh prefix behavior
            #
            # Suricata descriptions keep their original prefix.
            # Wazuh descriptions receive "Wazuh: ".
            if (
                source == "Wazuh"
                and not description.startswith("Wazuh: ")
            ):
                description = "Wazuh: " + description

            # -------------------------------------------------
            # Agent information
            # -------------------------------------------------
            #
            # IMPORTANT:
            #
            # This intentionally uses the same hostname lookup
            # behavior as the original code:
            #
            #     location = j["agent"]["ip"]
            #     host = ips[location]
            #
            # We do NOT use ips.get(..., "") here because that
            # would silently remove hostname information when
            # an IP is not found.
            # -------------------------------------------------

            agent = j.get("agent", {})

            agent_ip = agent.get("ip", "")

            # Preserve original hostname lookup behavior.
            #
            # If agent_ip is not present in the YAML mapping,
            # this raises KeyError just like the original code.
            host = ips[agent_ip]

            # -------------------------------------------------
            # Wazuh rule information
            # -------------------------------------------------

            rule = j.get("rule", {})

            rule_id = rule.get("id", "")
            level = rule.get("level", "")

            # -------------------------------------------------
            # Decoder
            # -------------------------------------------------

            decoder_data = j.get("decoder", {})

            if isinstance(decoder_data, dict):

                decoder = decoder_data.get(
                    "name",
                    "",
                )

            else:

                decoder = decoder_data

            # -------------------------------------------------
            # Log location
            # -------------------------------------------------

            location = j.get(
                "location",
                "",
            )

            # -------------------------------------------------
            # Network information
            # -------------------------------------------------

            src_ip = ""
            src_port = ""
            dst_ip = ""
            dst_port = ""
            protocol = ""

            # -------------------------------------------------
            # Try top-level fields first
            # -------------------------------------------------

            src_ip = j.get(
                "srcip",
                j.get(
                    "src_ip",
                    "",
                ),
            )

            src_port = j.get(
                "srcport",
                j.get(
                    "src_port",
                    "",
                ),
            )

            dst_ip = j.get(
                "dstip",
                j.get(
                    "dst_ip",
                    "",
                ),
            )

            dst_port = j.get(
                "dstport",
                j.get(
                    "dst_port",
                    "",
                ),
            )

            protocol = j.get(
                "protocol",
                "",
            )

            # -------------------------------------------------
            # Suricata/network information may be nested
            # under "data"
            # -------------------------------------------------

            data = j.get(
                "data",
                {},
            )

            if isinstance(data, dict):

                src_ip = data.get(
                    "srcip",
                    data.get(
                        "src_ip",
                        src_ip,
                    ),
                )

                src_port = data.get(
                    "srcport",
                    data.get(
                        "src_port",
                        src_port,
                    ),
                )

                dst_ip = data.get(
                    "dstip",
                    data.get(
                        "dst_ip",
                        dst_ip,
                    ),
                )

                dst_port = data.get(
                    "dstport",
                    data.get(
                        "dst_port",
                        dst_port,
                    ),
                )

                protocol = data.get(
                    "protocol",
                    protocol,
                )

            # -------------------------------------------------
            # Alert ID
            # -------------------------------------------------

            alert_id = j.get(
                "id",
                "",
            )

            # Some Wazuh structures may contain an ID
            # inside data.
            if not alert_id and isinstance(data, dict):

                alert_id = data.get(
                    "id",
                    "",
                )

            # -------------------------------------------------
            # Write alert
            # -------------------------------------------------

            write_alert(
                alerts_csv=alerts_csv,
                log_time=log_time,
                source=source,
                name=description,
                short=short,
                agent_ip=agent_ip,
                host=host,
                rule_id=rule_id,
                level=level,
                decoder=decoder,
                location=location,
                src_ip=src_ip,
                src_port=src_port,
                dst_ip=dst_ip,
                dst_port=dst_port,
                protocol=protocol,
                alert_id=alert_id,
            )

        # =====================================================
        # AMiner alerts
        # =====================================================

        for line in aminer_in:

            j = json.loads(line)

            # -------------------------------------------------
            # Timestamp
            # -------------------------------------------------

            log_time = (
                j["LogData"]["DetectionTimestamp"][-1]
            )

            # -------------------------------------------------
            # Alert description
            # -------------------------------------------------

            analysis_component = (
                j.get(
                    "AnalysisComponent",
                    {}
                ).get(
                    "AnalysisComponentName",
                    ""
                )
            )

            description = analysis_component

            short = get_short(description)

            # -------------------------------------------------
            # AMiner ID
            # -------------------------------------------------

            aminer_id = (
                j.get(
                    "AMiner",
                    {}
                ).get(
                    "ID",
                    ""
                )
            )

            # -------------------------------------------------
            # Host lookup
            # -------------------------------------------------
            #
            # Preserve the original AMiner behavior exactly:
            #
            #     location = j["AMiner"]["ID"]
            #     host = ips[location]
            #
            # Therefore a missing mapping raises KeyError
            # instead of silently producing an empty hostname.
            # -------------------------------------------------

            host = ips[aminer_id]

            # -------------------------------------------------
            # AMiner-specific fields
            # -------------------------------------------------

            source = "AMiner"

            # The AMiner ID is retained in agent_ip because
            # there is no Wazuh agent.ip field in the AMiner
            # alert structure used here.
            agent_ip = aminer_id

            rule_id = ""
            level = ""
            decoder = ""
            location = ""

            src_ip = ""
            src_port = ""
            dst_ip = ""
            dst_port = ""
            protocol = ""

            # -------------------------------------------------
            # Alert ID
            # -------------------------------------------------

            alert_id = (
                j.get("id", "")
                or j.get(
                    "AMiner",
                    {}
                ).get(
                    "ID",
                    ""
                )
            )

            # -------------------------------------------------
            # Write alert
            # -------------------------------------------------

            write_alert(
                alerts_csv=alerts_csv,
                log_time=log_time,
                source=source,
                name=description,
                short=short,
                agent_ip=agent_ip,
                host=host,
                rule_id=rule_id,
                level=level,
                decoder=decoder,
                location=location,
                src_ip=src_ip,
                src_port=src_port,
                dst_ip=dst_ip,
                dst_port=dst_port,
                protocol=protocol,
                alert_id=alert_id,
            )

    print("Wrote:", alertscsv)

Processing: russellmitchell
Wrote: ./alerts_csv/russellmitchell_alerts.csv
Processing: fox
Wrote: ./alerts_csv/fox_alerts.csv
Processing: harrison
Wrote: ./alerts_csv/harrison_alerts.csv
Processing: santos
Wrote: ./alerts_csv/santos_alerts.csv
Processing: shaw
Wrote: ./alerts_csv/shaw_alerts.csv
Processing: wardbeck
Wrote: ./alerts_csv/wardbeck_alerts.csv
Processing: wheeler
Wrote: ./alerts_csv/wheeler_alerts.csv
Processing: wilson
Wrote: ./alerts_csv/wilson_alerts.csv


### Below is the code block for stage 2

In [2]:
# =========================================================
# Standardize existing alert CSV files
# =========================================================

import pandas as pd


alert_csv_files = [
    "fox_alerts.csv",
    "harrison_alerts.csv",
    "russellmitchell_alerts.csv",
    "santos_alerts.csv",
    "shaw_alerts.csv",
    "wardbeck_alerts.csv",
    "wheeler_alerts.csv",
    "wilson_alerts.csv",
]


for filename in alert_csv_files:
    file_path = os.path.join(alerts_csv_path, filename)

    print(f"Processing {filename}...")

    # Read CSV
    df = pd.read_csv(file_path)

    # -----------------------------------------------------
    # Rename columns
    # -----------------------------------------------------
    df = df.rename(
        columns={
            "source": "name",
            "short": "short_abbreviation",
            "name": "short",
        }
    )

    # -----------------------------------------------------
    # Create standardized IP column
    #
    # src_ip takes precedence.
    # agent_ip is used when src_ip is null/empty.
    # -----------------------------------------------------
    df["ip"] = (
        df["src_ip"]
        .replace("", pd.NA)
        .fillna(df["agent_ip"])
    )

    # -----------------------------------------------------
    # Overwrite the original CSV
    # -----------------------------------------------------
    df.to_csv(file_path, index=False)

    print(f"  ✓ Overwritten: {file_path}")

Processing fox_alerts.csv...


/tmp/ipykernel_10124/1931273609.py:26: DtypeWarning: Columns (12,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  ✓ Overwritten: ./alerts_csv/fox_alerts.csv
Processing harrison_alerts.csv...


/tmp/ipykernel_10124/1931273609.py:26: DtypeWarning: Columns (8,9,10,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  ✓ Overwritten: ./alerts_csv/harrison_alerts.csv
Processing russellmitchell_alerts.csv...


...

  ✓ Overwritten: ./alerts_csv/wardbeck_alerts.csv
Processing wheeler_alerts.csv...


/tmp/ipykernel_10124/1931273609.py:26: DtypeWarning: Columns (12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  ✓ Overwritten: ./alerts_csv/wheeler_alerts.csv
Processing wilson_alerts.csv...


/tmp/ipykernel_10124/1931273609.py:26: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  ✓ Overwritten: ./alerts_csv/wilson_alerts.csv
